In [1]:
import numpy as np
import scipy
from pyscf import ao2mo, gto, scf, lo
#from functools import reduce
import sys
sys.path.append('../../../Hamiltonian')
import hamiltonian

# construct Hamiltonian

In [2]:
length = 2.0
angle = 104.5
nqubits = 14

In [3]:
mol = gto.Mole()
mol.atom = hamiltonian.genH2O(bond=length,angle=angle)
mol.basis = 'sto3g' 
mol.charge = 0
mol.spin = 0
mol.symmetry = False
mol.build()

# RHF
mf = scf.RHF(mol)
eRHF = mf.kernel()

ecore = mol.energy_nuc()
h1e = mf.get_hcore()
h2e = mf._eri

# use OAO orbitals
oao = lo.orth.orth_ao(mol, method='meta_lowdin', pre_orth_ao='ANO', s=None)
mo_coeff = oao

# Transform one- and two-electron integrals from the atomic orbital (AO) basis to the spin-orbital (SO) basis in ABAB order.
h1m,h2m = hamiltonian.get_ao2so_h1_h2_int(mo_coeff,h1e,h2e)
# convert a FermiOperator into Pauli strings and coefficients for NetKet operator construction.
qham = hamiltonian.get_qubit_hamiltonian(ecore,h1m,h2m,'jordan_wigner')
qham_compress = qham.compress()
ob_string_list, coeff_list = hamiltonian.generalop2pauliobAlocal_nk(qham_compress, nqubits)
np.save('string_op/ob_string_list_R_'+str(length)+'.npy', ob_string_list)
np.save('string_op/coeff_list_R_'+str(length)+'.npy', coeff_list)

# get one-particle term
qham_h1 = hamiltonian.get_qubit_hamiltonian(ecore*0.0,h1m,h2m*0.0,'jordan_wigner')
qham_h1_compress = qham_h1.compress()
ob_string_list_h1, coeff_list_h1 = hamiltonian.generalop2pauliobAlocal_nk(qham_h1_compress, nqubits)
np.save('string_op/ob_string_list_R_'+str(length)+'_h1.npy', ob_string_list_h1)
np.save('string_op/coeff_list_R_'+str(length)+'_h1.npy', coeff_list_h1)

# get two-particle term
qham_h2 = hamiltonian.get_qubit_hamiltonian(ecore*0.0,h1m*0.0,h2m,'jordan_wigner')
qham_h2_compress = qham_h2.compress()
ob_string_list_h2, coeff_list_h2 = hamiltonian.generalop2pauliobAlocal_nk(qham_h2_compress, nqubits) 
np.save('string_op/ob_string_list_R_'+str(length)+'_h2.npy', ob_string_list_h2)
np.save('string_op/coeff_list_R_'+str(length)+'_h2.npy', coeff_list_h2)

converged SCF energy = -74.4011724867711


# calculate the ground state energy and ground state wave function

In [4]:
import netket as nk

/home/chang/soft/miniconda3/envs/mindquantum_10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import jax

from netket.utils import struct
from netket.utils.types import Scalar, Array

from netket.hilbert.constraint import DiscreteHilbertConstraint


class ABConstraint(DiscreteHilbertConstraint):
    """
    Constraint of an Hilbert space enforcing the number of alpha and beta electrons in the degrees of freedom.
    """

    Na: Scalar = struct.field(pytree_node=False)
    Nb: Scalar = struct.field(pytree_node=False)

    def __init__(self, Na: Scalar, Nb: Scalar):
        if Na is None or Nb is None:
            raise TypeError("Na and Nb must be a number.")

        self.Na = Na
        self.Nb = Nb

    @jax.jit
    def __call__(self, x: Array) -> Array:
        return jax.numpy.logical_and((x*0.5+0.5)[..., 0::2].sum(axis=-1)== round(self.Na), (x*0.5+0.5)[..., 1::2].sum(axis=-1)== round(self.Nb))

    def __hash__(self):
        return hash(("ABConstraint", self.Na, self.Nb))

    def __eq__(self, other):
        if isinstance(other, ABConstraint):
            return jax.numpy.logical_and(self.Na == other.Na, self.Nb == other.Nb)
        return False

    def __repr__(self):
        return f"ABConstraint({self.Na, self.Nb})"

In [6]:
Na = Nb = 5
hilbert = nk.hilbert.Spin(0.5, nqubits, constraint=ABConstraint(Na=Na, Nb=Nb),inverted_ordering=True)
ob_string_list = np.load('string_op/ob_string_list_R_'+str(length)+'.npy')
coeff_list = np.load('string_op/coeff_list_R_'+str(length)+'.npy')
operator = nk.operator.PauliStrings(hilbert, ob_string_list, coeff_list)
op_mat = operator.to_sparse()

In [7]:
scipy.sparse.save_npz('ham/hamiltonian_R2.0_na_nb.npz', op_mat.real)

In [8]:
op_mat_dense = op_mat.real.todense()
ee, vv = scipy.linalg.eigh(op_mat_dense)
np.save('ee_vv/ee_R'+str(length)+'.npy', ee)
np.save('ee_vv/vv_R'+str(length)+'.npy', vv)

In [9]:
Pn = vv[:,0]**2
np.save('ee_vv/Pn_R'+str(length)+'.npy', Pn)

# calculate spectral gap

In [10]:
sys.path.append('../../../ProposalMat')
import ProposalMat

In [11]:
A_mat = ProposalMat.get_A(Pn) # the acceptance probability matrix A
idx = ProposalMat.get_sigmaAidx(nqubits, Na, Nb)  # the Hilbert space with Na = 5, Nb=5

## ExcitationSD proposal

In [12]:
Q_mat_EXSD = ProposalMat.get_Q_excitationSD_mat(idx, int(nqubits/2), Na, Nb)
P_mat_EXSD = ProposalMat.get_Pij(Q_mat_EXSD, A_mat)
ee_P, vv_P = scipy.sparse.linalg.eigs(P_mat_EXSD.T, k=3, which='LM')

In [13]:
ee_S, vv_S = ProposalMat.sort_eevv(ee_P, vv_P)
delta = ee_S[0] - ee_S[1]

In [14]:
delta

0.00913765811809164

## Uniform proposal

In [15]:
Q_mat_Uniform = ProposalMat.get_Q_uniform_mat(idx)
P_mat_Uniform = ProposalMat.get_Pij(Q_mat_Uniform, A_mat)
ee_P, vv_P = scipy.sparse.linalg.eigs(P_mat_Uniform.T, k=3, which='LM')

In [16]:
ee_S, vv_S = ProposalMat.sort_eevv(ee_P, vv_P)
delta = ee_S[0] - ee_S[1]

In [17]:
delta

0.007822414044019621

## Effective proposal 
$Q[i, j] = \sum_n p_{n}(\boldsymbol{S}_i)p_{n}(\boldsymbol{S}_j)$

In [18]:
Q_mat_Effective = ProposalMat.get_Q_effective_mat(vv)
P_mat_Effective = ProposalMat.get_Pij(Q_mat_Effective, A_mat)
ee_P, vv_P = scipy.sparse.linalg.eigs(P_mat_Effective.T, k=3, which='LM')

In [19]:
ee_S, vv_S = ProposalMat.sort_eevv(ee_P, vv_P)
delta = ee_S[0] - ee_S[1]

In [20]:
delta

3.419486915845482e-13

At this point, the Hamiltonian becomes block-diagonalized, satisfying the commutation relation $[\hat{H}, \hat{n}_{2\mathrm{P_{z\alpha}}}\hat{n}_{2\mathrm{P_{z\beta}}}+(1-\hat{n}_{2\mathrm{P_{z\alpha}}})(1-\hat{n}_{2\mathrm{P_{z\beta}}})]=0$

In [21]:
class ABConstraint_specify(DiscreteHilbertConstraint):
    """
    Constraint of an Hilbert space enforcing the number of alpha and beta electrons in the degrees of freedom.
    
    motify directly
    """

    Na: Scalar = struct.field(pytree_node=False)
    Nb: Scalar = struct.field(pytree_node=False)

    def __init__(self, Na: Scalar, Nb: Scalar):
        if Na is None or Nb is None:
            raise TypeError("Na and Nb must be a number.")

        self.Na = Na
        self.Nb = Nb

    @jax.jit
    def __call__(self, x: Array) -> Array:
        return jax.numpy.logical_and(jax.numpy.logical_and((x*0.5+0.5)[..., 0::2].sum(axis=-1)== round(self.Na), (x*0.5+0.5)[..., 1::2].sum(axis=-1)== round(self.Nb)), 
                                     2*(x*0.5+0.5)[...,8]*(x*0.5+0.5)[...,9]-(x*0.5+0.5)[...,8]-(x*0.5+0.5)[...,9]+1 == round(1))

    def __hash__(self):
        return hash(("ABConstraint_specify", self.Na, self.Nb))

    def __eq__(self, other):
        if isinstance(other, ABConstraint_specify):
            return jax.numpy.logical_and(self.Na == other.Na, self.Nb == other.Nb)
        return False

    def __repr__(self):
        return f"ABConstraint_specify({self.Na, self.Nb})"

In [22]:
hilbert_sub = nk.hilbert.Spin(0.5, nqubits, constraint=ABConstraint_specify(Na=Na, Nb=Nb),inverted_ordering=True)
operator_sub = nk.operator.PauliStrings(hilbert_sub, ob_string_list, coeff_list)
op_mat_sub = operator_sub.to_sparse().real

In [23]:
scipy.sparse.save_npz('ham/hamiltonian_R2.0_na_nb_2pz.npz', op_mat_sub.real)

In [24]:
op_mat_sub_dense = op_mat_sub.todense()
ee_sub, vv_sub = scipy.linalg.eigh(op_mat_sub_dense)
np.save('ee_vv/ee_R'+str(length)+'_sub.npy', ee_sub)
np.save('ee_vv/vv_R'+str(length)+'_sub.npy', vv_sub)

In [25]:
Pn_sub = vv_sub[:,0]**2
np.save('ee_vv/Pn_R'+str(length)+'_sub.npy', Pn_sub)

In [26]:
A_mat_sub = ProposalMat.get_A(Pn_sub)
Q_mat_Effective_sub = ProposalMat.get_Q_effective_mat(vv_sub)
P_mat_Effective_sub = ProposalMat.get_Pij(Q_mat_Effective_sub, A_mat_sub)
ee_P_sub, vv_P_sub = scipy.sparse.linalg.eigs(P_mat_Effective_sub.T, k=3, which='LM')

/home/chang/Desktop/work/MCMC/code/example/spectral_gap/H2O/../../../ProposalMat/ProposalMat.py:845: RuntimeWarning: divide by zero encountered in divide
  A_mat = np.einsum('i,j->ij', 1/Pn, Pn, optimize=True)


In [27]:
ee_S_sub, vv_S_sub = ProposalMat.sort_eevv(ee_P_sub, vv_P_sub)
delta = ee_S_sub[0] - ee_S_sub[1]

In [28]:
delta

4.8753713042692937e-05

## Quantum proposal
$Q[i, j] = |⟨i|e^{-\mathrm{i}\hat{H}t}|j⟩|^2$

In [29]:
delta_list = []
for time in np.arange(0.1, 40, 0.2):
    Q_mat_Quantum_sub = ProposalMat.get_Q_quantum_mat(ee_sub, vv_sub, time)
    P_mat_Quantum_sub = ProposalMat.get_Pij(Q_mat_Quantum_sub, A_mat_sub)
    ee_P_sub, vv_P_sub = scipy.sparse.linalg.eigs(P_mat_Quantum_sub.T, k=3, which='LM')
    ee_S_sub, vv_S_sub = ProposalMat.sort_eevv(ee_P_sub, vv_P_sub)
    delta = ee_S_sub[0] - ee_S_sub[1]
    delta_list.append(delta)

In [30]:
np.max(delta_list)

9.077819221070538e-05

## Effective (hopping) proposal
$\hat{H}(\gamma) = \big((1-\gamma)\sum_{\substack{pq,\sigma}} h_{pq} \hat{a}_{p\sigma}^\dagger \hat{a}_{q\sigma} +\gamma\alpha\hat{H}_{\mathrm{hopping}}\big)+\frac{1}{2} \sum_{\substack{pqrs,\sigma\tau}} g_{pqrs} \hat{a}_{p\sigma}^\dagger \hat{a}_{r\tau}^\dagger \hat{a}_{s\tau} \hat{a}_{q\sigma} + E_{\mathrm{nuc}}$,$\alpha=\frac{\lVert h\lVert_F}{\sqrt{2n(n-1)}}$

$\hat{H}_{\mathrm{hopping}}=-\sum_{\substack{pq, \sigma \\ p \neq q}}\hat{a}_{p\sigma}^\dagger \hat{a}_{q\sigma}$

$Q[i, j, \gamma] = \sum_n p_{n}(\boldsymbol{S}_i, \gamma)p_{n}(\boldsymbol{S}_j,\gamma)$

$Q^{\mathrm{eff(hopping)}}=\frac{1}{\gamma_1-\gamma_0}\int_{\gamma_0}^{\gamma_1} \mathrm{d} \gamma Q[i, j, \gamma] \approx \frac{1}{N+1}\sum_{k=0}^{N} Q[i, j, \gamma_0+k\triangle \gamma]$, where $\triangle \gamma = \frac{\gamma_1-\gamma_0}{N}$

get the hopping Hamiltonian and the normalizing factor

In [31]:
norb = int(nqubits/2)
thop = np.ones((norb,norb))-np.eye(norb)
b_hopping = np.zeros((norb,2*norb)) 
b_hopping[:,::2] = np.eye(norb).copy()
b_hopping[:,1::2] = np.eye(norb).copy()
h1_hopping = b_hopping.T.dot(thop).dot(b_hopping)
h1_hopping[::2,1::2]=h1_hopping[1::2,::2]=0.

qham_h1_hopping = hamiltonian.get_qubit_hamiltonian(ecore*0.0,h1_hopping,h2m*0.0,'jordan_wigner')
qham_h1_hopping_compress = qham_h1_hopping.compress()
ob_string_list_hopping, coeff_list_hopping = hamiltonian.generalop2pauliobAlocal_nk(qham_h1_hopping_compress, nqubits)
np.save('string_op/ob_string_list_R_'+str(length)+'_hopping.npy', ob_string_list_hopping)
np.save('string_op/coeff_list_R_'+str(length)+'_hopping.npy', coeff_list_hopping)

alpha = np.linalg.norm(h1m)/np.sqrt(2*norb*(norb-1))
operator_hopping = nk.operator.PauliStrings(hilbert, ob_string_list_hopping, coeff_list_hopping)
op_mat_hopping = operator_hopping.to_sparse()

In [32]:
ob_string_list_h1 = np.load('string_op/ob_string_list_R_'+str(length)+'_h1.npy')
coeff_list_h1 = np.load('string_op/coeff_list_R_'+str(length)+'_h1.npy')
operator_h1 = nk.operator.PauliStrings(hilbert, ob_string_list_h1, coeff_list_h1)
op_mat_h1 = operator_h1.to_sparse()

ob_string_list_h2 = np.load('string_op/ob_string_list_R_'+str(length)+'_h2.npy')
coeff_list_h2 = np.load('string_op/coeff_list_R_'+str(length)+'_h2.npy')
operator_h2 = nk.operator.PauliStrings(hilbert, ob_string_list_h2, coeff_list_h2)
op_mat_h2 = operator_h2.to_sparse()

In [33]:
gamma_list = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
gamma_length = len(gamma_list)
Q_mat_Effective_hopping = np.zeros((len(Pn), len(Pn)))
for gamma in gamma_list:
    op_hubb = op_mat_h1*(1-gamma) - op_mat_hopping * gamma * alpha + op_mat_h2
    ee, vv = scipy.linalg.eigh(op_hubb.todense())
    np.save('ee_vv/ee_R' + str(length) + '_hubb_gamma_' + str(gamma) + '_re_h1.npy', ee)
    np.save('ee_vv/vv_R' + str(length) + '_hubb_gamma_' + str(gamma) + '_re_h1.npy', vv)
    Q_mat_Effective_hopping += ProposalMat.get_Q_effective_mat(vv)
Q_mat_Effective_hopping /= gamma_length
np.save('Q_mat/Q_mat_Effective_hopping.npy', Q_mat_Effective_hopping)
P_mat_Effective_hopping = ProposalMat.get_Pij(Q_mat_Effective_hopping, A_mat)
ee_P, vv_P = scipy.sparse.linalg.eigs(P_mat_Effective_hopping.T, k=3, which='LM')
ee_S, vv_S = ProposalMat.sort_eevv(ee_P, vv_P)
delta = ee_S[0] - ee_S[1]

In [34]:
delta

0.024695631733001155

## Quantum (hopping) proposal
$Q[i, j, \gamma] = |⟨i|e^{-\mathrm{i}\hat{H}(\gamma)t}|j⟩|^2$

$Q^{\mathrm{Quantum(hopping)}}(t)=\frac{1}{\gamma_1-\gamma_0}\int_{\gamma_0}^{\gamma_1} \mathrm{d} \gamma Q[i, j, \gamma, t]\approx \frac{1}{N+1}\sum_{k=0}^{N} Q[i, j, \gamma_0+k\triangle \gamma, t]$, where $\triangle \gamma = \frac{\gamma_1-\gamma_0}{N}$

In [35]:
delta_list = []
for time in np.arange(0.1, 40.01, 0.2):
    Q_mat_Quantum_hopping = np.zeros((len(Pn), len(Pn)))
    for gamma in gamma_list:
        ee = np.load('ee_vv/ee_R' + str(length) + '_hubb_gamma_' + str(gamma) + '_re_h1.npy')
        vv = np.load('ee_vv/vv_R' + str(length) + '_hubb_gamma_' + str(gamma) + '_re_h1.npy')
        Q_mat_Quantum_hopping += ProposalMat.get_Q_quantum_mat(ee, vv, time)
        del ee, vv
    Q_mat_Quantum_hopping /= gamma_length
    P_mat_Quantum_hopping = ProposalMat.get_Pij(Q_mat_Quantum_hopping, A_mat)
    ee_P, vv_P = scipy.sparse.linalg.eigs(P_mat_Quantum_hopping.T, k=3, which='LM')
    ee_S, vv_S = ProposalMat.sort_eevv(ee_P, vv_P)
    delta = ee_S[0] - ee_S[1]
    delta_list.append(delta)

In [36]:
np.max(delta_list)

0.034733649082573814

In [37]:
seq = np.where(delta_list == np.max(delta_list))
time_max = np.arange(0.1, 40.01, 0.2)[seq]

In [38]:
time_max

array([18.3])

In [39]:
Q_mat_Quantum_hopping = np.zeros((len(Pn), len(Pn)))
for gamma in gamma_list:
    ee = np.load('ee_vv/ee_R' + str(length) + '_hubb_gamma_' + str(gamma) + '_re_h1.npy')
    vv = np.load('ee_vv/vv_R' + str(length) + '_hubb_gamma_' + str(gamma) + '_re_h1.npy')
    Q_mat_Quantum_hopping += ProposalMat.get_Q_quantum_mat(ee, vv, time_max[0])
    del ee, vv
Q_mat_Quantum_hopping /= gamma_length
np.save('Q_mat/Q_mat_Quantum_hopping.npy', Q_mat_Quantum_hopping)